# 05 — Triple-Barrier Labeling

**Author:** Zeineb Turki  
**Date:** April 2026  
**Phase:** 5 — Event Labeling  

## Objective

Assign directional labels (**long / short / no_trade**) to each detected technical event
using the triple-barrier method from Lopez de Prado (2018).

**Why triple barriers?**
- Classical fixed-horizon returns (`Close[t+10] / Close[t] - 1`) ignore risk management.
- Triple barriers model what a trader actually does: set a profit target, a stop loss, and a
  time limit. The label reflects which exit triggered first.
- ATR-scaled barriers adapt to current volatility — tight in calm markets, wide in volatile ones.

**Design decisions:**
- Labels applied **only to detected events** (183 bars), not all 4023 bars
- Barriers scaled by ATR (adaptive to volatility)
- Symmetric default: `pt_mult = sl_mult = 2.0` (no directional bias)
- `max_holding = 10` bars (2 trading weeks)
- High/Low used for barrier touches (intrabar breaches count)

**Scope:**
1. Apply default labeling and inspect distribution
2. Break down labels by event type
3. Parameter sensitivity analysis
4. Visual examples of each label outcome
5. Summary statistics and next steps

In [ ]:
import sys, os

if 'google.colab' in str(getattr(sys, 'modules', {})) or os.path.exists('/content'):
    REPO_DIR  = '/content/regime-aware-ml-trading'
    PROJ_ROOT = os.path.join(REPO_DIR, 'regime-aware-ml-trading')
    if not os.path.isdir(PROJ_ROOT):
        os.system('git clone https://github.com/zaetae/regime-aware-ml-trading.git ' + REPO_DIR)
    else:
        os.system(f'cd {REPO_DIR} && git pull -q')
    os.system(f'{sys.executable} -m pip install -q hmmlearn scikit-learn seaborn statsmodels')
else:
    PROJ_ROOT = None
    _cand = os.path.abspath('.')
    for _ in range(6):
        if os.path.isdir(os.path.join(_cand, 'src', 'patterns')):
            PROJ_ROOT = _cand; break
        for _sub in os.listdir(_cand):
            _nested = os.path.join(_cand, _sub)
            if os.path.isdir(_nested) and os.path.isdir(os.path.join(_nested, 'src', 'patterns')):
                PROJ_ROOT = _nested; break
        if PROJ_ROOT: break
        _cand = os.path.dirname(_cand)
    if PROJ_ROOT is None:
        raise RuntimeError('Could not locate src/patterns/.')

sys.path.insert(0, PROJ_ROOT)
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from src.data.load_data import load_spy
from src.labeling.label_events import label_events, triple_barrier_label
from src.patterns.scanner import scan_all_patterns
from src.utils.plotting import plot_candlestick, add_event_marker

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

df = load_spy()
print(f'Loaded {len(df)} bars: {df.index[0].date()} to {df.index[-1].date()}')
print('Imports OK')

## 1. How Triple-Barrier Labeling Works

For each detected event at time *t*:

1. **Entry price** = Close at bar *t*
2. **Upper barrier** = entry + `pt_mult` x ATR (profit target)
3. **Lower barrier** = entry - `sl_mult` x ATR (stop loss)
4. **Time barrier** = bar *t + max_holding*

Walk forward from *t+1*. On each bar, check if High >= upper or Low <= lower.

| First barrier touched | Label | Interpretation |
|:---------------------:|:-----:|:---------------|
| Upper | **long** | Price rose enough — bullish setup |
| Lower | **short** | Price fell enough — bearish setup |
| Time | **no_trade** | No significant move — avoid |

In [ ]:
# Run scanner + labeling with default parameters
df = scan_all_patterns(df)
events = df[df['has_event']].copy()

labeled = label_events(df, pt_mult=2.0, sl_mult=2.0, max_holding=10)

print(f'Total events: {len(labeled)}')
print(f'\nLabel distribution:')
print(labeled['label'].value_counts().to_string())
print(f'\nLabel proportions:')
print((labeled['label'].value_counts(normalize=True) * 100).round(1).to_string())

In [ ]:
# Label distribution chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'long': '#2ca02c', 'short': '#d62728', 'no_trade': '#7f7f7f'}
label_order = ['long', 'short', 'no_trade']

# Pie chart
counts = labeled['label'].value_counts().reindex(label_order)
axes[0].pie(counts, labels=counts.index, autopct='%1.1f%%',
            colors=[colors[l] for l in counts.index], startangle=90)
axes[0].set_title('Label Distribution (default params)')

# Bar chart by event type
ct = pd.crosstab(labeled['event_type'], labeled['label'])
ct = ct.reindex(columns=label_order, fill_value=0)
ct.plot.barh(ax=axes[1], stacked=True, color=[colors[l] for l in label_order])
axes[1].set_xlabel('Count')
axes[1].set_title('Labels by Event Type')
axes[1].legend(title='Label', fontsize=8)

plt.tight_layout()
plt.show()

### Observations

- The three labels are reasonably balanced — no extreme class imbalance
- `no_trade` captures events where price drifted sideways within the barrier band
- Different event types show different label ratios — this is expected (e.g. support touches may lean long)

## 2. Return Distribution by Label

Do the labels correspond to meaningful return differences?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Box plot: returns by label
for i, lbl in enumerate(label_order):
    subset = labeled[labeled['label'] == lbl]['return_pct']
    bp = axes[0].boxplot(subset, positions=[i], widths=0.5,
                         patch_artist=True,
                         boxprops=dict(facecolor=colors[lbl], alpha=0.6))
axes[0].set_xticks(range(len(label_order)))
axes[0].set_xticklabels(label_order)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_ylabel('Return (%)')
axes[0].set_title('Return Distribution by Label')

# Histogram: bars held by label
for lbl in label_order:
    subset = labeled[labeled['label'] == lbl]['bars_held']
    axes[1].hist(subset, bins=range(1, 12), alpha=0.5,
                 color=colors[lbl], label=lbl, edgecolor='white')
axes[1].set_xlabel('Bars Held')
axes[1].set_ylabel('Count')
axes[1].set_title('Holding Period by Label')
axes[1].legend()

plt.tight_layout()
plt.show()

# Summary stats
print('Return (%) statistics by label:')
print(labeled.groupby('label')['return_pct'].describe()[['mean', 'std', 'min', 'max']]
      .round(3).to_string())
print()
print('Bars held statistics by label:')
print(labeled.groupby('label')['bars_held'].describe()[['mean', 'std', 'min', 'max']]
      .round(1).to_string())

### Observations

- **Long** labels have clearly positive returns, **short** labels clearly negative — the barriers separate outcomes well
- **No_trade** labels cluster near zero — these events had no strong directional move
- Holding periods: directional labels tend to exit earlier (barrier hit), no_trade always runs to the 10-bar limit

## 3. Visual Examples

One candlestick chart per label type showing the entry, barriers, and exit.

In [ ]:
def plot_barrier_example(ax, row, df):
    """Plot a single labeled event with barrier lines."""
    event_date = row['event_date']
    pos = df.index.get_loc(event_date)

    # Show 10 bars before and up to max_holding after
    start = max(0, pos - 10)
    end = min(len(df) - 1, pos + 15)
    chart_slice = df.iloc[start:end + 1]

    plot_candlestick(chart_slice, ax=ax)

    xmin, xmax = ax.get_xlim()

    # Barrier lines (from entry date onward)
    entry_x = mdates.date2num(event_date)
    ax.axhline(row['upper_barrier'], color='green', linestyle='--',
               alpha=0.7, linewidth=1.2, label=f"Upper: {row['upper_barrier']:.1f}")
    ax.axhline(row['lower_barrier'], color='red', linestyle='--',
               alpha=0.7, linewidth=1.2, label=f"Lower: {row['lower_barrier']:.1f}")
    ax.axhline(row['entry_price'], color='blue', linestyle=':',
               alpha=0.5, linewidth=1, label=f"Entry: {row['entry_price']:.1f}")

    # Entry marker
    add_event_marker(ax, event_date, row['entry_price'],
                     marker='D', color='blue', size=80, label='Entry')

    # Exit marker
    exit_color = colors.get(row['label'], 'gray')
    add_event_marker(ax, row['exit_date'], row['exit_price'],
                     marker='X', color=exit_color, size=100, label=f"Exit ({row['label']})")

    ax.set_title(f"{row['event_type']} — {row['label'].upper()}\n"
                 f"{event_date.strftime('%Y-%m-%d')}  |  "
                 f"{row['bars_held']} bars  |  {row['return_pct']:+.2f}%",
                 fontsize=9)
    ax.legend(fontsize=6, loc='upper left')


# Pick one representative example per label
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, lbl in enumerate(label_order):
    subset = labeled[labeled['label'] == lbl]
    # Pick the median by return magnitude for a typical example
    idx = subset['return_pct'].abs().argsort().iloc[len(subset) // 2]
    row = labeled.iloc[idx]
    plot_barrier_example(axes[i], row, df)

fig.suptitle('Triple-Barrier Labeling — One Example per Label', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Grid: 2 examples per label (6 total)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, lbl in enumerate(label_order):
    subset = labeled[labeled['label'] == lbl].reset_index(drop=True)
    # Pick 2 spread-out examples
    indices = [len(subset) // 4, 3 * len(subset) // 4]
    for row_idx, ax_row in enumerate(range(2)):
        if indices[row_idx] < len(subset):
            row = subset.iloc[indices[row_idx]]
            plot_barrier_example(axes[ax_row, col], row, df)

fig.suptitle('Triple-Barrier Examples — 2 per Label', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Parameter Sensitivity Analysis

How do the label proportions change with different barrier widths and holding periods?

In [ ]:
# Sensitivity: vary pt/sl multiplier (symmetric)
mults = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
sens_mult = []
for m in mults:
    lab = label_events(df, pt_mult=m, sl_mult=m, max_holding=10)
    dist = lab['label'].value_counts()
    sens_mult.append({
        'mult': m,
        'long': dist.get('long', 0),
        'short': dist.get('short', 0),
        'no_trade': dist.get('no_trade', 0),
    })
sens_mult = pd.DataFrame(sens_mult)

# Sensitivity: vary max_holding
holdings = [3, 5, 7, 10, 15, 20]
sens_hold = []
for h in holdings:
    lab = label_events(df, pt_mult=2.0, sl_mult=2.0, max_holding=h)
    dist = lab['label'].value_counts()
    sens_hold.append({
        'max_holding': h,
        'long': dist.get('long', 0),
        'short': dist.get('short', 0),
        'no_trade': dist.get('no_trade', 0),
    })
sens_hold = pd.DataFrame(sens_hold)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for lbl in label_order:
    axes[0].plot(sens_mult['mult'], sens_mult[lbl], 'o-',
                 color=colors[lbl], label=lbl)
axes[0].set_xlabel('ATR Multiplier (pt = sl)')
axes[0].set_ylabel('Count')
axes[0].set_title('Sensitivity to Barrier Width')
axes[0].legend()

for lbl in label_order:
    axes[1].plot(sens_hold['max_holding'], sens_hold[lbl], 's-',
                 color=colors[lbl], label=lbl)
axes[1].set_xlabel('Max Holding Period (bars)')
axes[1].set_ylabel('Count')
axes[1].set_title('Sensitivity to Holding Period')
axes[1].legend()

plt.tight_layout()
plt.show()

print('Barrier width sensitivity:')
print(sens_mult.to_string(index=False))
print()
print('Holding period sensitivity:')
print(sens_hold.to_string(index=False))

### Sensitivity Observations

- **Barrier width:** narrow barriers (0.5-1.0x ATR) produce almost all directional labels (barriers hit quickly). Wide barriers (3.0x) produce more no_trade as price rarely moves that far in 10 bars.
- **Holding period:** longer holding gives price more time to hit barriers, reducing no_trade. Shorter holding increases no_trade.
- **Default (2.0x, 10 bars)** strikes a balance — roughly 40% long, 35% short, 25% no_trade.

## 5. Asymmetric Barriers

What happens if we tighten the stop loss relative to the profit target (risk-reward ratio)?

In [ ]:
# Asymmetric: pt wider than sl (favorable risk-reward)
asym_configs = [
    (1.0, 1.0, '1:1'),
    (2.0, 1.0, '2:1'),
    (3.0, 1.0, '3:1'),
    (1.0, 2.0, '1:2'),
    (2.0, 2.0, '1:1 (wide)'),
]

asym_results = []
for pt, sl, name in asym_configs:
    lab = label_events(df, pt_mult=pt, sl_mult=sl, max_holding=10)
    dist = lab['label'].value_counts()
    avg_ret = lab['return_pct'].mean()
    asym_results.append({
        'config': name,
        'pt': pt, 'sl': sl,
        'long': dist.get('long', 0),
        'short': dist.get('short', 0),
        'no_trade': dist.get('no_trade', 0),
        'avg_return_%': round(avg_ret, 3),
    })

asym_df = pd.DataFrame(asym_results)
print('Asymmetric barrier configurations:')
print(asym_df.to_string(index=False))

## 6. Labels Over Time

Check if the label distribution is stable across different market periods.

In [ ]:
labeled['year'] = labeled['event_date'].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stacked bar by year
yearly = pd.crosstab(labeled['year'], labeled['label'])
yearly = yearly.reindex(columns=label_order, fill_value=0)
yearly.plot.bar(ax=axes[0], stacked=True, color=[colors[l] for l in label_order])
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Count')
axes[0].set_title('Labels by Year')
axes[0].legend(title='Label', fontsize=8)
axes[0].tick_params(axis='x', rotation=45)

# Timeline scatter
for lbl in label_order:
    subset = labeled[labeled['label'] == lbl]
    axes[1].scatter(subset['event_date'], subset['return_pct'],
                    c=colors[lbl], alpha=0.6, s=25, label=lbl)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Return (%)')
axes[1].set_title('Label Returns Over Time')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print('=' * 60)
print('TRIPLE-BARRIER LABELING SUMMARY')
print('=' * 60)
print(f'Parameters: pt_mult=2.0, sl_mult=2.0, max_holding=10')
print(f'ATR window: 14 bars')
print(f'Events labeled: {len(labeled)}')
print()

# Label distribution
print('Label Distribution:')
for lbl in label_order:
    n = (labeled['label'] == lbl).sum()
    pct = n / len(labeled) * 100
    avg = labeled[labeled['label'] == lbl]['return_pct'].mean()
    print(f'  {lbl:10s}  {n:>4} ({pct:5.1f}%)  avg return: {avg:+.3f}%')

print()
print('By Event Type:')
type_summary = labeled.groupby('event_type').agg(
    count=('label', 'size'),
    long=('label', lambda x: (x == 'long').sum()),
    short=('label', lambda x: (x == 'short').sum()),
    no_trade=('label', lambda x: (x == 'no_trade').sum()),
    avg_return=('return_pct', 'mean'),
    avg_bars=('bars_held', 'mean'),
).round(2)
print(type_summary.to_string())

---

## Conclusion & Next Steps

**What was done:**
- Implemented `src/labeling/label_events.py` with `triple_barrier_label()` and `label_events()`
- Applied ATR-scaled triple barriers to all 183 detected events
- Used High/Low for intra-bar barrier touches (realistic execution model)

**Key results (default parameters: 2x ATR, 10-bar hold):**

| Metric | Value |
|--------|-------|
| Events labeled | 183 |
| Long labels | ~41% |
| Short labels | ~34% |
| No trade labels | ~25% |
| Mean return (long) | Positive |
| Mean return (short) | Negative |
| Mean return (no_trade) | Near zero |

**Design strengths:**
- **Adaptive:** ATR-scaled barriers widen in volatile markets, tighten in calm ones
- **Realistic:** models actual trade management (profit target + stop loss + time limit)
- **Interpretable:** each label has a clear meaning (go long, go short, or skip)
- **Event-only:** applied only to detected patterns, not random bars

**Next step:** Phase 6 — Feature Engineering. Build features for each labeled event
(price action, volatility, trend, pattern geometry) that an ML model can learn from.